# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**The contract, in plain words.**

- **One row** in `fact_content_daily_performance` is one content page on one `report_date`, a page-day. My lane scores pages, so I roll those page-days up to one row per page over the month.
- **Tables:** `fact_content_daily_performance` for activity, with `dim_content` and `dim_clients` on hand for page metadata and per-client history.
- **Time window:** I develop on a mid-panel month, `2026-03`. The final month (June 2026, the `_sample` table) stays sealed as a test month, because it is the natural outcome window of a past-to-future label and developing label logic there would peek at the answer.
- **What I predict or rank:** a refresh-priority ranking of pages. The proxy label is a page's impressions decline, where the forward-window daily average falls below 0.8 of the prior-window average.
- **Deliberately excluded:** the forward-window activity itself as a feature. It is only knowable after the decision moment. Section 2 lists the rest.

In [1]:
import os
from huggingface_hub import hf_hub_download
import duckdb, numpy as np, pandas as pd

REPO = "FlyRank/internship-warehouse"
march = hf_hub_download(REPO, "fact_content_daily_performance/month=2026-03/data_0.parquet", repo_type="dataset")

con = duckdb.connect()
F = f"read_parquet('{march}')"
con.sql(f"SELECT report_date, client_hash_id, content_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position FROM {F} LIMIT 3").df()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Feature** (knowable at the decision moment, built from the prior window): `imp_prior`, `clk_prior`, `pos_prior`, `active_days_prior`, `sess_prior`.
- **Label / proxy:** the forward-window decline. Computed from forward impressions, so it is never a feature.
- **Context:** `content_hash_id`, `client_hash_id`, `report_date`, `month`. For grouping, joining, and client-holdout splits.
- **Excluded:** forward-window impressions and clicks (outcome window); rows where `gsc_data_available` is false or `gsc_avg_position = 0`, which mean no data rather than a real rank; and the AI-provider breakdown or any product-decision flag.

In [2]:
feature = ["imp_prior", "clk_prior", "pos_prior", "active_days_prior", "sess_prior"]
label = ["is_declining (forward daily avg < 0.8 x prior daily avg, from gsc_impressions)"]
context = ["content_hash_id", "client_hash_id", "report_date", "month"]
excluded = ["imp_forward / clk_forward (outcome window)",
            "gsc_data_available IS FALSE, gsc_avg_position = 0 (no data)",
            "ai_chatgpt..ai_other breakdown, any product-decision flag"]

for name, cols in [("feature", feature), ("label", label), ("context", context), ("excluded", excluded)]:
    print(f"{name:9} {cols}")

feature   ['imp_prior', 'clk_prior', 'pos_prior', 'active_days_prior', 'sess_prior']
label     ['is_declining (forward daily avg < 0.8 x prior daily avg, from gsc_impressions)']
context   ['content_hash_id', 'client_hash_id', 'report_date', 'month']
excluded  ['imp_forward / clk_forward (outcome window)', 'gsc_data_available IS FALSE, gsc_avg_position = 0 (no data)', 'ai_chatgpt..ai_other breakdown, any product-decision flag']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three checks on `2026-03`: the grain holds, the row count and date span, and how many rows survive an availability filter. Then the five-feature frame and the deliberate-leak test.

**The five features, and why each is knowable at the decision moment (end of the prior window, day 20):**

- `imp_prior`: GSC impressions summed over days 1-20. Knowable because it uses only days on or before the decision.
- `clk_prior`: GSC clicks, days 1-20. Same prior window.
- `pos_prior`: mean GSC position on impressed days 1-20. Prior days only.
- `active_days_prior`: number of days 1-20 with any impression. Prior window only.
- `sess_prior`: GA4 sessions, days 1-20, where GA4 is available.

The trap then adds `imp_forward`, the outcome window, on purpose, watches the score jump, and drops it.

In [3]:
print("Q1 grain (one row per page-day):")
print(con.sql(f"SELECT COUNT(*) AS dup_groups FROM (SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c FROM {F} GROUP BY 1,2,3 HAVING c > 1)").df().to_string(index=False))

print("\nQ2 row count + date span:")
print(con.sql(f"SELECT COUNT(*) AS n_rows, COUNT(DISTINCT content_hash_id) AS pages, COUNT(DISTINCT client_hash_id) AS clients, MIN(report_date) AS first_day, MAX(report_date) AS last_day FROM {F}").df().to_string(index=False))

print("\nQ3 availability (rows surviving IS TRUE):")
print(con.sql(f"SELECT COUNT(*) AS all_rows, SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_rows, SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_rows FROM {F}").df().to_string(index=False))

Q1 grain (one row per page-day):


 dup_groups
          0

Q2 row count + date span:


 n_rows  pages  clients  first_day   last_day
9841378 331437       55 2026-03-01 2026-03-31

Q3 availability (rows surviving IS TRUE):
 all_rows  gsc_rows  ga4_rows
  9841378 3611061.0  413966.0


In [4]:
frame = con.sql(f"""
WITH d AS (SELECT *, EXTRACT(day FROM report_date) AS dom FROM {F} WHERE gsc_data_available IS TRUE),
agg AS (
  SELECT client_hash_id, content_hash_id,
         SUM(CASE WHEN dom <= 20 THEN gsc_impressions ELSE 0 END) AS imp_prior,
         SUM(CASE WHEN dom <= 20 THEN gsc_clicks ELSE 0 END) AS clk_prior,
         AVG(CASE WHEN dom <= 20 AND gsc_impressions > 0 THEN gsc_avg_position END) AS pos_prior,
         COUNT(DISTINCT CASE WHEN dom <= 20 AND gsc_impressions > 0 THEN report_date END) AS active_days_prior,
         SUM(CASE WHEN dom <= 20 THEN ga4_sessions ELSE 0 END) AS sess_prior,
         SUM(CASE WHEN dom >= 21 THEN gsc_impressions ELSE 0 END) AS imp_forward
  FROM d GROUP BY 1, 2)
SELECT * FROM agg WHERE imp_prior >= 100
""").df()

frame["is_declining"] = (frame["imp_forward"] / 11.0 < 0.8 * frame["imp_prior"] / 20.0).astype(int)
print("candidate pages:", len(frame), "  declining base rate:", round(frame["is_declining"].mean(), 3))
frame[["imp_prior", "clk_prior", "pos_prior", "active_days_prior", "sess_prior", "is_declining"]].head()

candidate pages: 87267   declining base rate: 0.318


,imp_prior,clk_prior,pos_prior,active_days_prior,sess_prior,is_declining
0,953.0,2.0,4.346288,20,4.0,0
1,524.0,3.0,5.914034,20,4.0,0
2,6162.0,8.0,12.352343,20,9.0,1
3,171.0,1.0,16.876202,18,1.0,1
4,961.0,1.0,2.987973,20,0.0,1


In [5]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

feats = ["imp_prior", "clk_prior", "pos_prior", "active_days_prior", "sess_prior"]
y = frame["is_declining"].values
groups = frame["client_hash_id"].values
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42).split(frame, y, groups))

def auc(cols):
    m = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1, class_weight="balanced")
    m.fit(frame.iloc[tr][cols].fillna(0), y[tr])
    return roc_auc_score(y[te], m.predict_proba(frame.iloc[te][cols].fillna(0))[:, 1])

print(f"base rate declining        = {y.mean():.3f}")
print(f"honest 5 features    AUC   = {auc(feats):.3f}")
print(f"+ imp_forward (leak) AUC   = {auc(feats + ['imp_forward']):.3f}")
print("drop imp_forward, keep the honest number.")

base rate declining        = 0.318


honest 5 features    AUC   = 0.617


+ imp_forward (leak) AUC   = 1.000
drop imp_forward, keep the honest number.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **One month is a snapshot, not history.** Per-client GSC and GA4 start dates differ, so a single month hides uneven history. `dim_clients.gsc_data_start` and `ga4_data_start` come first before any longer window.
- **GA4 is sparse here.** Only a small share of March rows carry `ga4_data_available IS TRUE`; the rest are zero-filled, not zero engagement. Filter on the flag rather than reading the zeros.
- **`gsc_avg_position = 0` means no position data**, not rank zero.
- **The `_sample` table is June 2026**, the final month, so it is a sealed test window and never a place to develop label logic.
- Everything here is observational. None of it proves a Google ranking factor.

In [6]:
print(con.sql(f"""
  SELECT ROUND(AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0 END), 3) AS ga4_available_share,
         ROUND(AVG(CASE WHEN gsc_data_available IS TRUE THEN 1.0 ELSE 0 END), 3) AS gsc_available_share,
         ROUND(AVG(CASE WHEN gsc_avg_position = 0 THEN 1.0 ELSE 0 END), 3) AS position_zero_share
  FROM {F}
""").df().to_string(index=False))

 ga4_available_share  gsc_available_share  position_zero_share
               0.042                0.367                0.017


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.